# Day 04 - Part 2: File Handling + Logging
### Python → Generative AI Engineer Journey

---
**Author:** Shaurab Kumar Jha  
**Date:** Day 4 of 90  
**Goal:** MNC-ready Python & Gen AI Engineer

| # | Topic |
|---|-------|
| 1 | File I/O - open(), modes, context managers |
| 2 | Text files - read, write, append, encoding |
| 3 | CSV - csv module, DictReader, DictWriter |
| 4 | JSON - json module, encode/decode, edge cases |
| 5 | Pickle - binary serialization |
| 6 | `os` module - environment, processes, dirs |
| 7 | `pathlib` - modern path handling (OOP style) |
| 8 | `shutil` - file operations |
| 9 | `logging` - production-grade logging |
| 10 | `file_utils.py` - industry-ready utility module |

Complete coverage. Every mode, every method.

---
# PART 1 - FILE I/O FUNDAMENTALS

## File Open Modes

| Mode | Meaning | File must exist? | Truncates? |
|------|---------|-----------------|------------|
| `'r'` | Read (default) | ✅ Yes | ❌ |
| `'w'` | Write | ❌ (creates) | ✅ Yes |
| `'a'` | Append | ❌ (creates) | ❌ |
| `'x'` | Exclusive create | ❌ (fails if exists) | ❌ |
| `'r+'` | Read + Write | ✅ Yes | ❌ |
| `'w+'` | Write + Read | ❌ (creates) | ✅ Yes |
| `'a+'` | Append + Read | ❌ (creates) | ❌ |

**Add `b` for binary mode:** `'rb'`, `'wb'`, `'ab'`

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.1  TEXT FILE - WRITE, READ, APPEND
# ═══════════════════════════════════════════════════════════════════════════════

import os

DEMO_FILE = "demo.txt"

# ── WRITE - creates or overwrites ─────────────────────────────────────────────
# Always use context manager (with) - auto-closes even on exception
with open(DEMO_FILE, 'w', encoding='utf-8') as f:
    f.write("Line 1: Hello, Python!\n")
    f.write("Line 2: File I/O is powerful.\n")
    f.write("Line 3: Always use context managers.\n")
    written = f.tell()    # Current position in bytes
    print(f"Written {written} bytes")

# ── READ - all at once ────────────────────────────────────────────────────────
with open(DEMO_FILE, 'r', encoding='utf-8') as f:
    content = f.read()         # Entire file as single string
print(f"\nfull read:\n{content}")

# ── READ - line by line (memory efficient for large files) ────────────────────
with open(DEMO_FILE, 'r', encoding='utf-8') as f:
    print("Line by line:")
    for i, line in enumerate(f, 1):
        print(f"  {i}: {line.rstrip()}")

# ── READ - all lines into list ────────────────────────────────────────────────
with open(DEMO_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()     # ["Line 1...\n", "Line 2...\n", ...]
print(f"\nreadlines: {len(lines)} lines")

# readline() - one line at a time manually
with open(DEMO_FILE, 'r') as f:
    first  = f.readline()     # First line
    second = f.readline()     # Second line
print(f"readline 1: {first.rstrip()!r}")
print(f"readline 2: {second.rstrip()!r}")

# ── APPEND - add to end ───────────────────────────────────────────────────────
with open(DEMO_FILE, 'a', encoding='utf-8') as f:
    f.write("Line 4: Appended later.\n")

with open(DEMO_FILE) as f:
    print(f"\nAfter append: {len(f.readlines())} lines total")

# ── WRITE multiple lines at once ─────────────────────────────────────────────
lines_to_write = [f"entry_{i}\n" for i in range(5)]
with open("multi.txt", 'w') as f:
    f.writelines(lines_to_write)   # No newlines added - must include in strings
print("writelines() done")

Written 93 bytes

full read:
Line 1: Hello, Python!
Line 2: File I/O is powerful.
Line 3: Always use context managers.

Line by line:
  1: Line 1: Hello, Python!
  2: Line 2: File I/O is powerful.
  3: Line 3: Always use context managers.

readlines: 3 lines
readline 1: 'Line 1: Hello, Python!'
readline 2: 'Line 2: File I/O is powerful.'

After append: 4 lines total
writelines() done


In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1.2  FILE OBJECT METHODS - tell, seek, truncate
# ═══════════════════════════════════════════════════════════════════════════════

with open(DEMO_FILE, 'r') as f:
    print(f"Position at start : {f.tell()}")
    first_line = f.readline()
    print(f"After readline()  : {f.tell()} bytes")
    f.seek(0)                   # Go back to start
    print(f"After seek(0)     : {f.tell()}")
    f.seek(0, 2)                # Seek to end (0 bytes from end)
    print(f"End of file       : {f.tell()} bytes")

# ── Encoding ──────────────────────────────────────────────────────────────────
# Always specify encoding='utf-8' for cross-platform compatibility
# Windows default is cp1252, Linux is utf-8 - bugs happen without explicit encoding

UNICODE_FILE = "unicode_demo.txt"
with open(UNICODE_FILE, 'w', encoding='utf-8') as f:
    f.write("Hindi: नमस्ते\n")
    f.write("Emoji: 🐍🤖💡\n")
    f.write("Math:  ∑ ∫ √ π \n")

with open(UNICODE_FILE, 'r', encoding='utf-8') as f:
    print(f.read())

# ── errors parameter ─────────────────────────────────────────────────────────
# 'strict' (default) - raise UnicodeDecodeError
# 'ignore'           - skip bad bytes
# 'replace'          - replace bad bytes with ?
# 'backslashreplace' - replace with \xNN escape

with open(UNICODE_FILE, 'r', encoding='ascii', errors='replace') as f:
    degraded = f.read()
print(f"ASCII with replace: {degraded[:40]!r}...")

Position at start : 0
After readline()  : 24 bytes
After seek(0)     : 0
End of file       : 118 bytes
Hindi: नमस्ते
Emoji: 🐍🤖💡
Math:  ∑ ∫ √ π 

ASCII with replace: 'Hindi: ������������������\nEmoji: �������'...


---
# PART 2 - CSV FILES

CSV = Comma-Separated Values. Universal format for tabular data - Excel, databases, pandas all use it.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.1  CSV - writer, reader, DictWriter, DictReader
# ═══════════════════════════════════════════════════════════════════════════════

import csv

CSV_FILE = "employees.csv"

employees = [
    {"id": 1, "name": "Alice Sharma",  "role": "ML Engineer",    "salary": 120000, "city": "Bangalore"},
    {"id": 2, "name": "Bob Verma",     "role": "Data Analyst",   "salary": 85000,  "city": "Mumbai"},
    {"id": 3, "name": "Carol Singh",   "role": "DevOps Engineer","salary": 100000, "city": "Pune"},
    {"id": 4, "name": "Dave Kumar",    "role": "ML Engineer",    "salary": 135000, "city": "Hyderabad"},
    {"id": 5, "name": "Eve Patel,PhD", "role": "Research Sci.",  "salary": 180000, "city": "Delhi"},  # comma in name!
]

# ── DictWriter - write list of dicts ─────────────────────────────────────────
fieldnames = ["id", "name", "role", "salary", "city"]

with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
    # newline='' is REQUIRED on Windows to prevent double newlines
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()              # Write column headers
    writer.writerows(employees)       # Write all rows at once
    # writer.writerow(employees[0])   # Write single row

print("CSV written. Contents:")
with open(CSV_FILE) as f:
    print(f.read())

# ── DictReader - read into list of dicts ─────────────────────────────────────
with open(CSV_FILE, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    # reader.fieldnames contains headers
    data = list(reader)               # Each row is an OrderedDict

print(f"Read {len(data)} rows")
print(f"Columns: {list(data[0].keys())}")
for row in data:
    print(f"  {row['id']}: {row['name']:<20} {row['role']:<20} ₹{int(row['salary']):,}")

CSV written. Contents:
id,name,role,salary,city
1,Alice Sharma,ML Engineer,120000,Bangalore
2,Bob Verma,Data Analyst,85000,Mumbai
3,Carol Singh,DevOps Engineer,100000,Pune
4,Dave Kumar,ML Engineer,135000,Hyderabad
5,"Eve Patel,PhD",Research Sci.,180000,Delhi

Read 5 rows
Columns: ['id', 'name', 'role', 'salary', 'city']
  1: Alice Sharma         ML Engineer          ₹120,000
  2: Bob Verma            Data Analyst         ₹85,000
  3: Carol Singh          DevOps Engineer      ₹100,000
  4: Dave Kumar           ML Engineer          ₹135,000
  5: Eve Patel,PhD        Research Sci.        ₹180,000


In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.2  CSV - advanced options, dialects, quoting
# ═══════════════════════════════════════════════════════════════════════════════

# ── Custom delimiter (TSV - tab-separated) ────────────────────────────────────
TSV_FILE = "data.tsv"
with open(TSV_FILE, 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(["Name", "Score", "Grade"])
    writer.writerows([("Alice", 95, "A+"), ("Bob", 82, "A")])

# ── csv.writer - write lists (not dicts) ─────────────────────────────────────
PIPE_FILE = "data_pipe.csv"
with open(PIPE_FILE, 'w', newline='') as f:
    writer = csv.writer(f,
                        delimiter='|',
                        quotechar='"',
                        quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["id", "description", "price"])
    writer.writerow([1, "Laptop Pro, 16GB", 75000])  # comma in description
    writer.writerow([2, 'Cable "HDMI" 2m',  999])    # quotes in description

with open(PIPE_FILE) as f:
    print("Pipe-delimited CSV:")
    print(f.read())

# ── csv.reader - read lists ───────────────────────────────────────────────────
with open(PIPE_FILE) as f:
    reader = csv.reader(f, delimiter='|')
    headers = next(reader)             # Skip header row
    print(f"Headers: {headers}")
    for row in reader:
        print(f"  Row: {row}")

# ── csv quoting constants ─────────────────────────────────────────────────────
print("\nCSV Quoting modes:")
for name in ['QUOTE_ALL', 'QUOTE_MINIMAL', 'QUOTE_NONNUMERIC', 'QUOTE_NONE']:
    print(f"  csv.{name} = {getattr(csv, name)}")

# ── Sniff dialect - auto-detect format ───────────────────────────────────────
with open(CSV_FILE) as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample)
    print(f"\nDetected delimiter: {dialect.delimiter!r}")

Pipe-delimited CSV:
id|description|price
1|Laptop Pro, 16GB|75000
2|"Cable ""HDMI"" 2m"|999

Headers: ['id', 'description', 'price']
  Row: ['1', 'Laptop Pro, 16GB', '75000']
  Row: ['2', 'Cable "HDMI" 2m', '999']

CSV Quoting modes:
  csv.QUOTE_ALL = 1
  csv.QUOTE_MINIMAL = 0
  csv.QUOTE_NONNUMERIC = 2
  csv.QUOTE_NONE = 3

Detected delimiter: ','


---
# PART 3 - JSON FILES

JSON is the universal language of APIs.  
Every Gen AI API (OpenAI, Groq, Anthropic) sends/receives JSON.

In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.1  JSON - loads, dumps, load, dump + all options
# ═══════════════════════════════════════════════════════════════════════════════

import json
from datetime import datetime

# Python ↔ JSON type mapping:
# Python dict  ↔ JSON object {}
# Python list  ↔ JSON array  []
# Python str   ↔ JSON string
# Python int/float ↔ JSON number
# Python True/False ↔ JSON true/false
# Python None  ↔ JSON null
# Python tuple → JSON array (loses tuple type on decode)

data = {
    "user": {"id": 101, "name": "Alice", "active": True, "score": None},
    "skills": ["Python", "ML", "Docker"],
    "meta": {"version": 2, "created": "2024-01-01"}
}

# ── json.dumps - Python dict → JSON string ────────────────────────────────────
compact    = json.dumps(data)                           # Compact (default)
pretty     = json.dumps(data, indent=2)                 # Human-readable
sorted_key = json.dumps(data, indent=2, sort_keys=True) # Sorted keys
ascii_safe = json.dumps({"hindi": "नमस्ते"}, ensure_ascii=True)   # Escape non-ASCII
unicode_ok = json.dumps({"hindi": "नमस्ते"}, ensure_ascii=False)  # Keep unicode

print(f"compact    : {compact[:60]}...")
print(f"ascii_safe : {ascii_safe}")
print(f"unicode_ok : {unicode_ok}")
print(f"\npretty:\n{pretty}")

# ── json.loads - JSON string → Python dict ────────────────────────────────────
json_str = '{"name": "Bob", "scores": [85, 92, 78], "passed": true, "note": null}'
parsed   = json.loads(json_str)
print(f"\nparsed type: {type(parsed)}")       # dict
print(f"parsed: {parsed}")
print(f"true→{type(parsed['passed'])}, null→{type(parsed['note'])}")

# ── json.dump - write to file ─────────────────────────────────────────────────
JSON_FILE = "/tmp/employees.json"
with open(JSON_FILE, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"\nJSON written to {JSON_FILE}")

# ── json.load - read from file ────────────────────────────────────────────────
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    loaded = json.load(f)
print(f"Loaded: {loaded['user']}")

compact    : {"user": {"id": 101, "name": "Alice", "active": true, "score...
ascii_safe : {"hindi": "\u0928\u092e\u0938\u094d\u0924\u0947"}
unicode_ok : {"hindi": "नमस्ते"}

pretty:
{
  "user": {
    "id": 101,
    "name": "Alice",
    "active": true,
    "score": null
  },
  "skills": [
    "Python",
    "ML",
    "Docker"
  ],
  "meta": {
    "version": 2,
    "created": "2024-01-01"
  }
}

parsed type: <class 'dict'>
parsed: {'name': 'Bob', 'scores': [85, 92, 78], 'passed': True, 'note': None}
true→<class 'bool'>, null→<class 'NoneType'>


FileNotFoundError: [Errno 2] No such file or directory: '/tmp/employees.json'

In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3.2  JSON - custom encoder/decoder for non-serializable types
# ═══════════════════════════════════════════════════════════════════════════════

from datetime import datetime, date
from decimal import Decimal
import json

# Problem: datetime, Decimal, custom objects are not JSON-serializable
order = {
    "order_id": "ORD001",
    "created": datetime.now(),        # Not serializable!
    "total": Decimal("1999.99"),      # Not serializable!
}

try:
    json.dumps(order)
except TypeError as e:
    print(f"Default serialization fails: {e}")

# ── Solution 1: default parameter ────────────────────────────────────────────
def json_default(obj):
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()          # datetime → "2024-01-15T09:30:00"
    if isinstance(obj, Decimal):
        return float(obj)               # Decimal → float
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

serialized = json.dumps(order, default=json_default, indent=2)
print(f"\nWith default fn:\n{serialized}")

# ── Solution 2: Custom JSONEncoder class ──────────────────────────────────────
class EnhancedEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (datetime, date)):
            return {"_type": "datetime", "value": obj.isoformat()}
        if isinstance(obj, Decimal):
            return {"_type": "decimal", "value": str(obj)}
        if isinstance(obj, set):
            return list(obj)
        return super().default(obj)


class EnhancedDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        super().__init__(object_hook=self._decode, *args, **kwargs)

    def _decode(self, d):
        if d.get("_type") == "datetime":
            return datetime.fromisoformat(d["value"])
        if d.get("_type") == "decimal":
            return Decimal(d["value"])
        return d


encoded = json.dumps(order, cls=EnhancedEncoder, indent=2)
decoded = json.loads(encoded, cls=EnhancedDecoder)

print(f"\nCustom encoded:\n{encoded}")
print(f"\nDecoded types:")
print(f"  created: {type(decoded['created']).__name__} = {decoded['created']}")
print(f"  total  : {type(decoded['total']).__name__} = {decoded['total']}")

Default serialization fails: Object of type datetime is not JSON serializable

With default fn:
{
  "order_id": "ORD001",
  "created": "2026-04-09T23:50:21.445925",
  "total": 1999.99
}

Custom encoded:
{
  "order_id": "ORD001",
  "created": {
    "_type": "datetime",
    "value": "2026-04-09T23:50:21.445925"
  },
  "total": {
    "_type": "decimal",
    "value": "1999.99"
  }
}

Decoded types:
  created: datetime = 2026-04-09 23:50:21.445925
  total  : Decimal = 1999.99


---
# PART 4 - PICKLE - Binary Serialization

In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.1  PICKLE - serialize ANY Python object
# ═══════════════════════════════════════════════════════════════════════════════

import pickle

# Pickle can serialize: any Python object - classes, functions, lambdas, models
# JSON can only handle: dict, list, str, int, float, bool, None

data = {
    "model": lambda x: x * 2,          # Lambda - not JSON serializable
    "matrix": [[1,2,3],[4,5,6]],
    "config": {"lr": 0.001, "epochs": 50},
    "history": (1, 2, 3),               # Tuple preserved
}

PICKLE_FILE = "data.pkl"

# ── Serialize to file ─────────────────────────────────────────────────────────
with open(PICKLE_FILE, 'wb') as f:      # 'wb' - write binary
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Pickled to {PICKLE_FILE}")

# ── Deserialize from file ─────────────────────────────────────────────────────
with open(PICKLE_FILE, 'rb') as f:      # 'rb' - read binary
    loaded = pickle.load(f)

print(f"Loaded matrix    : {loaded['matrix']}")
print(f"Loaded tuple type: {type(loaded['history']).__name__}")   # tuple preserved!
print(f"Lambda works     : {loaded['model'](21)}")

# ── In-memory pickle ─────────────────────────────────────────────────────────
serialized   = pickle.dumps(data)      # → bytes
deserialized = pickle.loads(serialized)
print(f"\nIn-memory pickle size: {len(serialized)} bytes")

# ── SECURITY WARNING ─────────────────────────────────────────────────────────
print("""
⚠️  PICKLE SECURITY WARNING:
   NEVER unpickle data from untrusted sources.
   Pickle can execute arbitrary code on load.
   Use JSON for data exchange. Use Pickle only for:
   - Caching ML models (sklearn, numpy arrays)
   - Session data within your own system
   - Temp files you created yourself
""")

PicklingError: Can't pickle <function <lambda> at 0x000001F694CEB380>: attribute lookup <lambda> on __main__ failed

---
# PART 5 - os MODULE - Complete Coverage

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5.1  os MODULE - everything
# ═══════════════════════════════════════════════════════════════════════════════

import os

# ── System info ───────────────────────────────────────────────────────────────
print(f"OS name         : {os.name}")
print(f"Platform sep    : {os.sep!r}")
print(f"Path separator  : {os.pathsep!r}")
print(f"Line separator  : {os.linesep!r}")
print(f"CPU count       : {os.cpu_count()}")

# ── Environment variables ────────────────────────────────────────────────────
print(f"\nPATH (first 60) : {os.environ.get('PATH', '')[:60]}...")
print(f"HOME            : {os.environ.get('HOME', 'N/A')}")
print(f"USER            : {os.environ.get('USER', 'N/A')}")

# Set environment variable
os.environ['MY_APP_DEBUG'] = 'true'
print(f"MY_APP_DEBUG    : {os.environ.get('MY_APP_DEBUG')}")

# ── Directory operations ──────────────────────────────────────────────────────
print(f"\nCurrent dir     : {os.getcwd()}")

DEMO_DIR = "/tmp/demo_folder"
os.makedirs(DEMO_DIR, exist_ok=True)    # Create dir + parents, no error if exists
os.makedirs(f"{DEMO_DIR}/sub/deep", exist_ok=True)
print(f"Created dirs    : {DEMO_DIR}")

# Create demo files
for name in ["file1.txt", "file2.txt", "report.csv", "data.json"]:
    open(f"{DEMO_DIR}/{name}", 'w').close()

# ── List directory ────────────────────────────────────────────────────────────
print(f"\nos.listdir()    : {os.listdir(DEMO_DIR)}")

# os.walk - recursive directory traversal
print("\nos.walk:")
for root, dirs, files in os.walk(DEMO_DIR):
    level = root.replace(DEMO_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

# ── os.path - old style path operations ──────────────────────────────────────
path = "/home/user/projects/app/main.py"
print(f"\nos.path operations on: {path}")
print(f"dirname         : {os.path.dirname(path)}")
print(f"basename        : {os.path.basename(path)}")
print(f"splitext        : {os.path.splitext(path)}")
print(f"split           : {os.path.split(path)}")
print(f"join            : {os.path.join('/home/user', 'docs', 'readme.txt')}")
print(f"exists          : {os.path.exists(DEMO_FILE)}")
print(f"isfile          : {os.path.isfile(DEMO_FILE)}")
print(f"isdir           : {os.path.isdir(DEMO_DIR)}")
print(f"getsize         : {os.path.getsize(DEMO_FILE)} bytes")
print(f"abspath         : {os.path.abspath('.')}") 
print(f"expanduser ~    : {os.path.expanduser('~/.bashrc')}")

OS name         : nt
Platform sep    : '\\'
Path separator  : ';'
Line separator  : '\r\n'
CPU count       : 8

PATH (first 60) : c:\Users\shaur\AppData\Local\Programs\Python\Python313;c:\Us...
HOME            : N/A
USER            : N/A
MY_APP_DEBUG    : true

Current dir     : d:\python-genai-journey\day04_Error_handling_Modules_File_Handlings
Created dirs    : /tmp/demo_folder

os.listdir()    : ['data.json', 'file1.txt', 'file2.txt', 'report.csv', 'sub']

os.walk:
demo_folder/
  data.json
  file1.txt
  file2.txt
  report.csv
  sub/
    deep/

os.path operations on: /home/user/projects/app/main.py
dirname         : /home/user/projects/app
basename        : main.py
splitext        : ('/home/user/projects/app/main', '.py')
split           : ('/home/user/projects/app', 'main.py')
join            : /home/user\docs\readme.txt
exists          : True
isfile          : True
isdir           : True
getsize         : 118 bytes
abspath         : d:\python-genai-journey\day04_Error_handling_Modu

---
# PART 6 - pathlib - Modern Path Handling

`pathlib` is the modern, OOP replacement for `os.path`. Use it in all new code.

In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.1  pathlib.Path - complete coverage
# ═══════════════════════════════════════════════════════════════════════════════

from pathlib import Path

# ── Creating Path objects ─────────────────────────────────────────────────────
p  = Path("/tmp/demo_folder")
home = Path.home()                      # User's home directory
cwd  = Path.cwd()                       # Current working directory

print(f"p       = {p}")
print(f"home    = {home}")
print(f"cwd     = {cwd}")

# ── Path arithmetic - / operator ─────────────────────────────────────────────
config = home / ".config" / "myapp" / "settings.json"
print(f"\npath /: {config}")

# ── Path properties ───────────────────────────────────────────────────────────
f = Path("/home/alice/projects/genai/day04.py")
print(f"\nPath: {f}")
print(f"  .name      = {f.name}")
print(f"  .stem      = {f.stem}")
print(f"  .suffix    = {f.suffix}")
print(f"  .suffixes  = {f.suffixes}")
print(f"  .parent    = {f.parent}")
print(f"  .parents   = {list(f.parents)}")
print(f"  .parts     = {f.parts}")
print(f"  .anchor    = {f.anchor}")
print(f"  .is_absolute() = {f.is_absolute()}")

# ── Check existence ───────────────────────────────────────────────────────────
demo = Path(DEMO_FILE)
print(f"\n{DEMO_FILE}:")
print(f"  .exists()  = {demo.exists()}")
print(f"  .is_file() = {demo.is_file()}")
print(f"  .is_dir()  = {demo.is_dir()}")
print(f"  .stat().st_size = {demo.stat().st_size} bytes")

# ── Reading and writing ───────────────────────────────────────────────────────
p2 = Path("/tmp/pathlib_test.txt")
p2.write_text("Hello from pathlib!\nLine 2\nLine 3", encoding='utf-8')
content = p2.read_text(encoding='utf-8')
print(f"\nread_text:\n{content}")

p3 = Path("/tmp/binary_test.bin")
p3.write_bytes(b"\x00\x01\x02\x03")
print(f"read_bytes: {p3.read_bytes()}")

# ── Directory operations ──────────────────────────────────────────────────────
new_dir = Path("/tmp/pathlib_demo")
new_dir.mkdir(parents=True, exist_ok=True)

# Create files
for name in ["a.txt", "b.py", "c.csv", "d.json"]:
    (new_dir / name).touch()        # Create empty file

# ── Listing files ─────────────────────────────────────────────────────────────
print("\niterimdir:")
for item in new_dir.iterdir():
    print(f"  {item.name} | is_file={item.is_file()}")

# glob - pattern matching
print("\nglob('*.txt'):")
for path in new_dir.glob("*.txt"):
    print(f"  {path}")

print("\nrglob('**/*.py') - recursive:")
for path in new_dir.rglob("*.py"):
    print(f"  {path}")

# ── Rename / resolve ──────────────────────────────────────────────────────────
a_file = new_dir / "a.txt"
renamed = a_file.rename(new_dir / "a_renamed.txt")
print(f"\nRenamed: {a_file.name} → {renamed.name}")
print(f"Resolved absolute: {renamed.resolve()}")

# with_name, with_stem, with_suffix
sample = Path("/data/report_2024.csv")
print(f"\nwith_suffix   : {sample.with_suffix('.json')}")
print(f"with_stem     : {sample.with_stem('final_report')}")
print(f"with_name     : {sample.with_name('output.xlsx')}")

p       = \tmp\demo_folder
home    = C:\Users\shaur
cwd     = d:\python-genai-journey\day04_Error_handling_Modules_File_Handlings

path /: C:\Users\shaur\.config\myapp\settings.json

Path: \home\alice\projects\genai\day04.py
  .name      = day04.py
  .stem      = day04
  .suffix    = .py
  .suffixes  = ['.py']
  .parent    = \home\alice\projects\genai
  .parents   = [WindowsPath('/home/alice/projects/genai'), WindowsPath('/home/alice/projects'), WindowsPath('/home/alice'), WindowsPath('/home'), WindowsPath('/')]
  .parts     = ('\\', 'home', 'alice', 'projects', 'genai', 'day04.py')
  .anchor    = \
  .is_absolute() = False

demo.txt:
  .exists()  = True
  .is_file() = True
  .is_dir()  = False
  .stat().st_size = 118 bytes

read_text:
Hello from pathlib!
Line 2
Line 3
read_bytes: b'\x00\x01\x02\x03'

iterimdir:
  a.txt | is_file=True
  b.py | is_file=True
  c.csv | is_file=True
  d.json | is_file=True

glob('*.txt'):
  \tmp\pathlib_demo\a.txt

rglob('**/*.py') - recursive:
  \tmp\path

In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6.2  shutil - high-level file operations
# ═══════════════════════════════════════════════════════════════════════════════

import shutil
from pathlib import Path

src  = Path("/tmp/demo.txt")
dst  = Path("/tmp/demo_copy.txt")
dst2 = Path("/tmp/shutil_demo/")

# Copy file
shutil.copy2(src, dst)            # copy2 preserves metadata
print(f"Copied: {src} → {dst}")

# Copy entire directory tree
if dst2.exists(): shutil.rmtree(dst2)
shutil.copytree("/tmp/demo_folder", dst2)
print(f"Copied dir: {dst2}")

# Move file
moved = shutil.move(str(dst), "/tmp/moved_demo.txt")
print(f"Moved to: {moved}")

# Disk usage
usage = shutil.disk_usage("/tmp")
print(f"\n/tmp disk:")
print(f"  Total : {usage.total / 1e9:.1f} GB")
print(f"  Used  : {usage.used  / 1e9:.1f} GB")
print(f"  Free  : {usage.free  / 1e9:.1f} GB")

# ── Cleanup ───────────────────────────────────────────────────────────────────
# shutil.rmtree(dst2)   # Remove directory tree
# os.remove(src)        # Remove single file
# os.rmdir(dir)         # Remove EMPTY directory

FileNotFoundError: [WinError 2] The system cannot find the file specified

---
# PART 7 - LOGGING - Production-Grade

## Why not `print()`?

| Feature | `print()` | `logging` |
|---------|-----------|----------|
| Levels (DEBUG/INFO/WARNING/ERROR) | ❌ | ✅ |
| Output to file | ❌ | ✅ |
| Timestamp & line numbers | ❌ | ✅ |
| Turn off in production | ❌ | ✅ |
| Multiple handlers | ❌ | ✅ |
| Log rotation | ❌ | ✅ |

## Log Levels (lowest to highest severity)
```
DEBUG    (10) ← development details
INFO     (20) ← normal operation
WARNING  (30) ← something unexpected
ERROR    (40) ← error that needs attention
CRITICAL (50) ← application cannot continue
```

In [17]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.1  LOGGING - basicConfig, levels, handlers, formatters
# ═══════════════════════════════════════════════════════════════════════════════

import logging
import sys
from pathlib import Path

LOG_FILE = Path("/tmp/app.log")

# ── Reset root logger (needed in notebooks where it may be configured already)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# ── basicConfig - simplest setup ─────────────────────────────────────────────
logging.basicConfig(
    level   = logging.DEBUG,           # Minimum level to capture
    format  = "%(asctime)s | %(levelname)-8s | %(name)s:%(lineno)d | %(message)s",
    datefmt = "%Y-%m-%d %H:%M:%S",
    handlers= [
        logging.StreamHandler(sys.stdout),              # Console
        logging.FileHandler(LOG_FILE, encoding='utf-8') # File
    ]
)

logger = logging.getLogger(__name__)   # Best practice: named logger

logger.debug("Debug: only shown in development")
logger.info("Info: application started")
logger.warning("Warning: config file missing, using defaults")
logger.error("Error: failed to connect to database")
logger.critical("Critical: disk full - application cannot continue")

# Log with extra context
user_id = 42
logger.info("User logged in", extra={"user_id": user_id})

# Log exceptions with traceback
try:
    result = 1 / 0
except ZeroDivisionError:
    logger.exception("Calculation failed")  # Automatically includes traceback

print(f"\nLog file written to: {LOG_FILE}")
print(f"Log file size: {LOG_FILE.stat().st_size} bytes")

2026-04-10 00:11:33 | DEBUG    | __main__:28 | Debug: only shown in development
2026-04-10 00:11:33 | INFO     | __main__:29 | Info: application started
2026-04-10 00:11:33 | WARNING  | __main__:30 | Warning: config file missing, using defaults
2026-04-10 00:11:33 | ERROR    | __main__:31 | Error: failed to connect to database
2026-04-10 00:11:33 | CRITICAL | __main__:32 | Critical: disk full - application cannot continue
2026-04-10 00:11:33 | INFO     | __main__:36 | User logged in
2026-04-10 00:11:33 | ERROR    | __main__:42 | Calculation failed
Traceback (most recent call last):
  File "C:\Users\shaur\AppData\Local\Temp\ipykernel_10516\2412609600.py", line 40, in <module>
    result = 1 / 0
             ~~^~~
ZeroDivisionError: division by zero

Log file written to: \tmp\app.log
Log file size: 770 bytes


In [18]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7.2  PRODUCTION LOGGING SETUP - multiple handlers, rotation, structured logs
# ═══════════════════════════════════════════════════════════════════════════════

import logging
import logging.handlers
import json
from datetime import datetime

# ── JSON log formatter - for ELK/Splunk/CloudWatch ───────────────────────────
class JSONFormatter(logging.Formatter):
    """Formats log records as JSON - easy to parse in log aggregation tools."""

    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp" : datetime.utcnow().isoformat() + "Z",
            "level"     : record.levelname,
            "logger"    : record.name,
            "module"    : record.module,
            "line"      : record.lineno,
            "message"   : record.getMessage(),
        }
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry)


def setup_logger(name: str, log_file: str = None, level: int = logging.DEBUG) -> logging.Logger:
    """
    Production-ready logger factory.
    Creates a named logger with console + optional rotating file handler.
    """
    log = logging.getLogger(name)
    log.setLevel(level)

    # Avoid adding handlers multiple times in notebook environment
    if log.handlers:
        return log

    # ── Console handler - human-readable ─────────────────────────────────────
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)    # Only INFO+ to console
    console_handler.setFormatter(logging.Formatter(
        "%(asctime)s [%(levelname)-8s] %(name)s - %(message)s",
        datefmt="%H:%M:%S"
    ))
    log.addHandler(console_handler)

    # ── Rotating file handler - max 5MB, keep 3 backups ─────────────────────
    if log_file:
        file_handler = logging.handlers.RotatingFileHandler(
            log_file,
            maxBytes   = 5 * 1024 * 1024,  # 5 MB
            backupCount= 3,
            encoding   = 'utf-8'
        )
        file_handler.setLevel(logging.DEBUG)  # DEBUG+ to file
        file_handler.setFormatter(JSONFormatter())
        log.addHandler(file_handler)

    return log


# Use the factory
app_log = setup_logger("myapp", log_file="/tmp/app_prod.log")

app_log.debug("Loading configuration...")
app_log.info("Application starting up")
app_log.info("Connected to database: users_db")
app_log.warning("Cache miss rate above 30%")
app_log.error("Payment gateway timeout")

# ── Logger hierarchy - parent-child ──────────────────────────────────────────
# child logger propagates to parent by default
api_log  = logging.getLogger("myapp.api")
db_log   = logging.getLogger("myapp.database")
api_log.info("GET /api/users 200")      # Uses myapp's handlers
db_log.warning("Slow query: 2.3s")

# Check JSON log file
json_log = Path("/tmp/app_prod.log")
if json_log.exists():
    print("\nJSON log entries:")
    for line in json_log.read_text().strip().split('\n')[:3]:
        print(f"  {line}")

2026-04-10 00:11:38 | DEBUG    | myapp:67 | Loading configuration...
00:11:38 [INFO    ] myapp - Application starting up
2026-04-10 00:11:38 | INFO     | myapp:68 | Application starting up
00:11:38 [INFO    ] myapp - Connected to database: users_db
2026-04-10 00:11:38 | INFO     | myapp:69 | Connected to database: users_db
00:11:38 [WARNING ] myapp - Cache miss rate above 30%
2026-04-10 00:11:38 | WARNING  | myapp:70 | Cache miss rate above 30%
00:11:38 [ERROR   ] myapp - Payment gateway timeout
2026-04-10 00:11:38 | ERROR    | myapp:71 | Payment gateway timeout
00:11:38 [INFO    ] myapp.api - GET /api/users 200
2026-04-10 00:11:38 | INFO     | myapp.api:77 | GET /api/users 200
00:11:38 [WARNING ] myapp.database - Slow query: 2.3s
2026-04-10 00:11:38 | WARNING  | myapp.database:78 | Slow query: 2.3s

JSON log entries:
  {"timestamp": "2026-04-09T18:26:38.127868Z", "level": "DEBUG", "logger": "myapp", "module": "763609894", "line": 67, "message": "Loading configuration..."}
  {"timestam

C:\Users\shaur\AppData\Local\Temp\ipykernel_10516\763609894.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp" : datetime.utcnow().isoformat() + "Z",


---
# PART 8 - file_utils.py - Industry-Ready Utility

This is what the GitHub file looks like - production code with proper error handling, logging, type hints, and docstrings.

In [19]:
# ═══════════════════════════════════════════════════════════════════════════════
# file_utils.py - PRODUCTION-READY FILE UTILITY MODULE
# This is exactly what goes in your GitHub repo
# ═══════════════════════════════════════════════════════════════════════════════

"""
file_utils.py
~~~~~~~~~~~~~
Production-ready utilities for CSV, JSON, and text file operations.
Includes proper error handling, logging, type hints, and docstrings.
"""

import csv
import json
import logging
import logging.handlers
import os
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Optional, Union


# ─── Logger Setup ─────────────────────────────────────────────────────────────
def _get_logger(name: str = "file_utils") -> logging.Logger:
    """Get or create a named logger for this module."""
    log = logging.getLogger(name)
    if not log.handlers:
        log.setLevel(logging.DEBUG)
        fmt = logging.Formatter(
            "%(asctime)s [%(levelname)-8s] %(name)s:%(lineno)d - %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        log.addHandler(sh)
    return log


logger = _get_logger()


# ─── CSV Utilities ────────────────────────────────────────────────────────────

def read_csv(
    path: Union[str, Path],
    encoding: str = "utf-8",
    delimiter: str = ",",
    skip_empty_rows: bool = True,
) -> list[dict]:
    """
    Read a CSV file into a list of dicts.

    Args:
        path: Path to CSV file.
        encoding: File encoding. Default 'utf-8'.
        delimiter: Field delimiter. Default ','.
        skip_empty_rows: Skip rows where all values are empty.

    Returns:
        List of row dicts (keys = column headers).

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file is empty or has no headers.
    """
    path = Path(path)
    logger.info(f"Reading CSV: {path}")

    if not path.exists():
        logger.error(f"CSV file not found: {path}")
        raise FileNotFoundError(f"No such file: {path}")

    if path.stat().st_size == 0:
        logger.warning(f"CSV file is empty: {path}")
        raise ValueError(f"File is empty: {path}")

    rows = []
    try:
        with open(path, "r", encoding=encoding, newline="") as f:
            reader = csv.DictReader(f, delimiter=delimiter)
            if not reader.fieldnames:
                raise ValueError(f"CSV has no header row: {path}")
            for i, row in enumerate(reader, start=2):  # start=2 (row 1 = headers)
                if skip_empty_rows and all(v.strip() == "" for v in row.values()):
                    logger.debug(f"Skipping empty row {i}")
                    continue
                rows.append(dict(row))
    except UnicodeDecodeError as e:
        logger.error(f"Encoding error reading {path}: {e}")
        raise

    logger.info(f"Read {len(rows)} rows from {path.name}")
    return rows


def write_csv(
    data: list[dict],
    path: Union[str, Path],
    fieldnames: Optional[list[str]] = None,
    encoding: str = "utf-8",
    delimiter: str = ",",
) -> int:
    """
    Write a list of dicts to a CSV file.

    Args:
        data: List of row dicts to write.
        path: Destination file path.
        fieldnames: Column order. Inferred from first row if None.
        encoding: File encoding.
        delimiter: Field delimiter.

    Returns:
        Number of rows written.

    Raises:
        ValueError: If data is empty.
    """
    if not data:
        raise ValueError("Cannot write empty data to CSV.")

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = fieldnames or list(data[0].keys())

    logger.info(f"Writing {len(data)} rows to {path}")

    try:
        with open(path, "w", encoding=encoding, newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
                extrasaction="ignore"  # Ignore extra keys in data
            )
            writer.writeheader()
            writer.writerows(data)
    except PermissionError as e:
        logger.error(f"Permission denied writing to {path}: {e}")
        raise

    logger.info(f"CSV written successfully: {path.name} ({path.stat().st_size:,} bytes)")
    return len(data)


# ─── JSON Utilities ───────────────────────────────────────────────────────────

class _DateTimeEncoder(json.JSONEncoder):
    """Extended JSON encoder - handles datetime objects."""
    def default(self, obj: Any) -> Any:
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, set):
            return sorted(list(obj))
        return super().default(obj)


def read_json(
    path: Union[str, Path],
    encoding: str = "utf-8",
) -> Any:
    """
    Read and parse a JSON file.

    Args:
        path: Path to JSON file.
        encoding: File encoding.

    Returns:
        Parsed Python object (dict, list, etc.).

    Raises:
        FileNotFoundError: If file not found.
        json.JSONDecodeError: If file contains invalid JSON.
    """
    path = Path(path)
    logger.info(f"Reading JSON: {path}")

    if not path.exists():
        logger.error(f"JSON file not found: {path}")
        raise FileNotFoundError(f"No such file: {path}")

    try:
        with open(path, "r", encoding=encoding) as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON in {path}: {e}")
        raise

    logger.info(f"JSON loaded: {path.name} → {type(data).__name__}")
    return data


def write_json(
    data: Any,
    path: Union[str, Path],
    indent: int = 2,
    ensure_ascii: bool = False,
    encoding: str = "utf-8",
) -> int:
    """
    Serialise data to a JSON file.

    Args:
        data: Python object to serialise.
        path: Destination file path.
        indent: Pretty-print indentation (None for compact).
        ensure_ascii: If False, preserve Unicode characters.
        encoding: File encoding.

    Returns:
        Number of bytes written.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    logger.info(f"Writing JSON to {path}")

    try:
        with open(path, "w", encoding=encoding) as f:
            json.dump(data, f, indent=indent,
                      ensure_ascii=ensure_ascii,
                      cls=_DateTimeEncoder)
    except TypeError as e:
        logger.error(f"JSON serialisation error: {e}")
        raise

    size = path.stat().st_size
    logger.info(f"JSON written: {path.name} ({size:,} bytes)")
    return size


# ─── Text Utilities ───────────────────────────────────────────────────────────

def read_text(
    path: Union[str, Path],
    encoding: str = "utf-8",
) -> str:
    """Read entire text file and return as string."""
    path = Path(path)
    logger.debug(f"Reading text: {path}")
    return path.read_text(encoding=encoding)


def write_text(
    text: str,
    path: Union[str, Path],
    encoding: str = "utf-8",
    mode: str = "w",
) -> int:
    """Write text to file. Returns bytes written."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, mode, encoding=encoding) as f:
        f.write(text)
    size = path.stat().st_size
    logger.debug(f"Text written: {path.name} ({size:,} bytes)")
    return size


def safe_open(
    path: Union[str, Path],
    mode: str = "r",
    encoding: str = "utf-8",
    default: Any = None,
):
    """
    Context manager that returns default value on FileNotFoundError.
    Use when a missing file is OK (config loading, optional cache, etc.).

    Usage:
        with safe_open('config.json', default={}) as f:
            if f: config = json.load(f)
    """
    from contextlib import contextmanager

    @contextmanager
    def _open():
        try:
            with open(path, mode, encoding=encoding if 'b' not in mode else None) as f:
                yield f
        except FileNotFoundError:
            logger.warning(f"File not found (returning default): {path}")
            yield default

    return _open()


# ─── Utility Functions ────────────────────────────────────────────────────────

def ensure_dir(path: Union[str, Path]) -> Path:
    """Create directory and all parents if they don't exist."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def file_info(path: Union[str, Path]) -> dict:
    """Return metadata dict for a file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No such file: {path}")
    stat = path.stat()
    return {
        "path"      : str(path.resolve()),
        "name"      : path.name,
        "stem"      : path.stem,
        "extension" : path.suffix,
        "size_bytes": stat.st_size,
        "size_kb"   : round(stat.st_size / 1024, 2),
        "is_file"   : path.is_file(),
        "is_dir"    : path.is_dir(),
        "modified"  : datetime.fromtimestamp(stat.st_mtime).isoformat(),
    }


# ─── Main - demo ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(" file_utils.py - DEMO")
    print("=" * 60)

    OUTPUT_DIR = Path("/tmp/file_utils_demo")
    ensure_dir(OUTPUT_DIR)

    # ── Sample dataset ────────────────────────────────────────────────────────
    sample_employees = [
        {"id": 1, "name": "Alice",  "dept": "Engineering", "salary": 120000, "joined": "2022-03-01"},
        {"id": 2, "name": "Bob",    "dept": "Marketing",   "salary": 85000,  "joined": "2023-01-15"},
        {"id": 3, "name": "Carol",  "dept": "Engineering", "salary": 100000, "joined": "2021-07-10"},
        {"id": 4, "name": "Dave",   "dept": "HR",          "salary": 75000,  "joined": "2023-09-01"},
        {"id": 5, "name": "Eve",    "dept": "Engineering", "salary": 150000, "joined": "2020-01-01"},
    ]

    # 1. Write CSV
    csv_path = OUTPUT_DIR / "employees.csv"
    rows_written = write_csv(sample_employees, csv_path)
    print(f"\n[CSV] Wrote {rows_written} rows")

    # 2. Read CSV back
    loaded_csv = read_csv(csv_path)
    print(f"[CSV] Read back {len(loaded_csv)} rows")

    # 3. Filter engineering team
    engineers = [r for r in loaded_csv if r["dept"] == "Engineering"]
    print(f"[CSV] Engineers: {[e['name'] for e in engineers]}")

    # 4. Write JSON
    report = {
        "generated": datetime.now(),    # Uses custom encoder
        "total_employees": len(loaded_csv),
        "by_dept": {},
        "avg_salary": 0,
    }
    for row in loaded_csv:
        dept = row["dept"]
        report["by_dept"].setdefault(dept, {"count": 0, "total_salary": 0})
        report["by_dept"][dept]["count"]        += 1
        report["by_dept"][dept]["total_salary"] += int(row["salary"])

    for dept, stats in report["by_dept"].items():
        stats["avg_salary"] = stats["total_salary"] // stats["count"]

    report["avg_salary"] = sum(int(r["salary"]) for r in loaded_csv) // len(loaded_csv)

    json_path = OUTPUT_DIR / "report.json"
    bytes_written = write_json(report, json_path)
    print(f"[JSON] Report written: {bytes_written:,} bytes")

    # 5. Read JSON back
    loaded_json = read_json(json_path)
    print(f"[JSON] Total employees: {loaded_json['total_employees']}")
    print(f"[JSON] Avg salary: ₹{loaded_json['avg_salary']:,}")

    for dept, stats in loaded_json["by_dept"].items():
        print(f"  {dept:<15}: {stats['count']} people | avg ₹{stats['avg_salary']:,}")

    # 6. File info
    print(f"\n[INFO] CSV file info:")
    info = file_info(csv_path)
    for k, v in info.items():
        print(f"  {k:<15}: {v}")

    # 7. safe_open demo
    print("\n[SAFE] Reading non-existent optional config:")
    with safe_open(OUTPUT_DIR / "optional_config.json", default=None) as f:
        config = json.load(f) if f else {"default_mode": True}
    print(f"  Config: {config}")

    print("\n" + "=" * 60)
    print(" All operations completed successfully!")
    print("=" * 60)

 file_utils.py - DEMO
2026-04-10 00:11:53 [INFO    ] file_utils:127 - Writing 5 rows to \tmp\file_utils_demo\employees.csv
2026-04-10 00:11:53 | INFO     | file_utils:127 | Writing 5 rows to \tmp\file_utils_demo\employees.csv
2026-04-10 00:11:53 [INFO    ] file_utils:142 - CSV written successfully: employees.csv (205 bytes)
2026-04-10 00:11:53 | INFO     | file_utils:142 | CSV written successfully: employees.csv (205 bytes)

[CSV] Wrote 5 rows
2026-04-10 00:11:53 [INFO    ] file_utils:68 - Reading CSV: \tmp\file_utils_demo\employees.csv
2026-04-10 00:11:53 | INFO     | file_utils:68 | Reading CSV: \tmp\file_utils_demo\employees.csv
2026-04-10 00:11:53 [INFO    ] file_utils:93 - Read 5 rows from employees.csv
2026-04-10 00:11:53 | INFO     | file_utils:93 | Read 5 rows from employees.csv
[CSV] Read back 5 rows
[CSV] Engineers: ['Alice', 'Carol', 'Eve']
2026-04-10 00:11:53 [INFO    ] file_utils:218 - Writing JSON to \tmp\file_utils_demo\report.json
2026-04-10 00:11:53 | INFO     | file_u

---
# Day 4 Complete Summary

## File Handling Cheatsheet
| Format | Module | Read | Write |
|--------|--------|------|-------|
| Text | `open()` | `.read()` / iterate | `.write()` |
| CSV | `csv` | `DictReader` | `DictWriter` |
| JSON | `json` | `json.load(f)` | `json.dump(d, f)` |
| Binary | `open('rb')` | `.read()` | `.write()` |
| Pickle | `pickle` | `pickle.load(f)` | `pickle.dump(o, f)` |

## Always Remember
- `encoding='utf-8'` on every file open
- `newline=''` in CSV operations on Windows  
- Use `pathlib.Path` over `os.path` in new code
- Use `logging` over `print()` in production
- Never pickle untrusted data

---